<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>Hugging Face 라이브러리(Hugging Face Transformers)</strong>에 대해 학습합니다.</br></br>
>AutoTokenizer, AutoModel, pipeline API로 사전학습 모델을 학습해봅시다.

</br>

# Hugging Face Transformers
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">사전학습된 NLP 모델을 쉽게 사용</mark>할 수 있게 해주는 라이브러리입니다.

처음부터 모델을 학습하는 것은 현실적으로 불가능합니다. BERT-base 학습에는 TPU v3 64개로 4일, GPT-3 학습에는 수백만 달러의 비용이 들었습니다. 따라서 대규모 텍스트 코퍼스로 범용 언어 표현을 미리 학습(Pre-training)한 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">공개된 사전 학습 모델을 재사용</mark>하는 것이 유일한 현실적 선택입니다. 이렇게 학습된 모델은 문법, 상식, 사실 지식까지 내재화되어 있어, 소량의 태스크 데이터로 파인튜닝(Fine-tuning)만 해도 월등한 성능을 얻을 수 있습니다.</br>

Hugging Face가 사실상 표준이 된 이유는 생태계의 통일 덕분입니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Model Hub</mark>에서 BERT, GPT, T5, LLaMA 등 수십만 개의 체크포인트(학습된 가중치 파일)를 다운로드할 수 있고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Auto 클래스</mark>로 모델명 하나만 지정하면 토크나이저, 모델, 설정이 자동으로 로드됩니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">pipeline API</mark>를 사용하면 단 한 줄로 감성 분석, 번역, 생성 등을 실행할 수 있으며, 논문 저자들이 직접 가중치를 업로드하여 재현성까지 보장됩니다. 결과적으로 Hugging Face는 NLP 연구와 실무 모두에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">공통 언어</mark>가 되었습니다.

</br>

## AutoTokenizer
> 모델에 맞는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">토크나이저를 자동으로 선택</mark>하고 로드합니다.

In [ ]:
# TODO 1: 사전학습된 토크나이저를 불러와 "Hello, world!"를 인코딩하여 input_ids, attention_mask, 토큰 목록, 디코딩 결과를 출력해봅시다.

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

encoded = tokenizer("Hello, world!", return_tensors="pt")
print(f"input_ids: {encoded['input_ids']}")
print(f"attention_mask: {encoded['attention_mask']}")
print(f"토큰: {tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])}")

decoded = tokenizer.decode(encoded["input_ids"][0])
print(f"디코딩: {decoded}")

</br>

## AutoModel

In [ ]:
# TODO 2: 사전학습된 시퀀스 분류 모델을 불러와 모델 타입과 파라미터 수를 출력해봅시다.

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=3
)
print(f"모델 타입: {type(model).__name__}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">Auto 클래스</th>
      <th>용도</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>AutoModel</code></td><td>기본 특성 추출</td></tr>
    <tr><td style="text-align:center"><code>AutoModelForSequenceClassification</code></td><td>텍스트 분류</td></tr>
    <tr><td style="text-align:center"><code>AutoModelForTokenClassification</code></td><td>개체명 인식 (NER)</td></tr>
    <tr><td style="text-align:center"><code>AutoModelForCausalLM</code></td><td>텍스트 생성</td></tr>
  </tbody>
</table>

</br>

## Pipeline API
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">한 줄로 추론</mark>을 수행하는 고수준 API입니다.

In [ ]:
# TODO 3: 감성 분석 파이프라인을 생성하고, "I love this movie!"를 입력하여 결과를 출력해봅시다.

from transformers import pipeline

classifier = pipeline("sentiment-analysis")
result = classifier("I love this movie!")
print(result)

In [ ]:
# TODO 4: 텍스트 생성 파이프라인을 만들고, "Once upon a time"을 입력하여 텍스트를 생성해봅시다.

generator = pipeline("text-generation", model="gpt2")
result = generator("Once upon a time", max_length=30, num_return_sequences=1)
print(result[0]["generated_text"])

💡pipeline 태스크 종류
> `sentiment-analysis`, `translation`, `text-generation`, `summarization`,</br>
> `question-answering`, `fill-mask`, `ner`, `zero-shot-classification` 등

</br>

## 추론 패턴 (Manual Pipeline)

In [ ]:
# TODO 5: 토크나이저로 "This is great!"를 인코딩한 뒤, 추론 모드에서 모델에 입력하여 로짓, 소프트맥스 확률, 예측 클래스를 출력해봅시다.

import torch

inputs = tokenizer("This is great!", return_tensors="pt", padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
probs = torch.softmax(logits, dim=-1)
predicted = torch.argmax(logits, dim=-1)

print(f"Logits: {logits.data.round(decimals=3)}")
print(f"확률: {probs.data.round(decimals=3)}")
print(f"예측 클래스: {predicted.item()}")

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">클래스</th>
      <th style="text-align:center">Logit</th>
      <th style="text-align:center">확률</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">0</td><td style="text-align:center">0.234</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">0.412</mark></td></tr>
    <tr><td style="text-align:center">1</td><td style="text-align:center">-0.156</td><td style="text-align:center">0.279</td></tr>
    <tr><td style="text-align:center">2</td><td style="text-align:center">0.089</td><td style="text-align:center">0.309</td></tr>
  </tbody>
</table>

💡pipeline vs 수동 추론
> pipeline: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">빠른 프로토타이핑</mark>, 기본 설정으로 충분할 때</br>
> 수동 추론: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">커스텀 후처리</mark>, 배치 처리, 성능 최적화가 필요할 때